In [1]:
%load_ext autoreload
%autoreload 2

import networkx as nx
from pathlib import Path
import pipeline
import pandas as pd
from collections import defaultdict
from geopy.distance import distance
import pickle
import numpy as np

### build initial full network (if needed)

In [2]:
data = Path('data/undir_trials/24h/depnet_undirected_24h.gz')
filepath = Path('data/undir_trials/24h/full_network_24h.txt')
if not filepath.exists():
    df = pd.read_csv(data)
    df['DEP'] = pd.to_numeric(df['DEP'], errors='coerce')
    df = df.dropna(subset=['DEP'])
    df.to_csv(filepath, sep=' ', index=False, header=False)
    print(f"Saved with {len(df)} edges.")
else:
    print("Network already exists.")

Network already exists.


load in

In [ ]:
gpath = Path('data/undir_trials/24h/full_network_24h.txt')
G = pipeline.load(gpath)

# --- add census tract attribute ---
pipeline.node_to_area(G)
num_empty = sum(1 for n, d in G.nodes(data=True) if d.get('tract') == 'Unknown')
print(f'% empty: {((num_empty/len(G.nodes()))*100):.4f}')

# --- add time/income attrs ---
pipeline.add_outside_metadata(G)

### map POIs to buckets (creating skeleton)

In [ ]:
# --- create blank network ---

# collect cats and tracts
cats = set(nx.get_node_attributes(G, 'poi_type').values())
tracts = set(nx.get_node_attributes(G, 'tract').values())
agg_nodes = {f"{c}||{t}" for c in cats for t in tracts}

print(f'len cats: {len(cats)} -- len tracts: {len(tracts)} -- # combos: {len(agg_nodes)}')

# init network
G_agg = nx.Graph()
G_agg.add_nodes_from(agg_nodes)

# --- map nodes to buckets ---

mapping = defaultdict(list)

for poi, attrs in G.nodes(data=True):
    c = attrs.get('poi_type')
    t = attrs.get('tract')
    key = f"{c}||{t}"

    if key in G_agg:
        mapping[key].append(poi)


# --- remove nodes that don't meet visit threshold or are empty ---

visit_threshold = 50
nodes_to_remove = set()

# sum up visits (node attr)
for bucket, poi_list in mapping.items():
    total_visits = sum(G.nodes[poi].get('total_visits', 0) for poi in poi_list)
    if total_visits >= visit_threshold:
        G_agg.nodes[bucket]['total_visits'] = total_visits
    else:
        nodes_to_remove.add(bucket)

# include empty buckets
for node in G_agg.nodes():
    if node not in mapping:
        nodes_to_remove.add(node)

G_agg.remove_nodes_from(nodes_to_remove)

print(f"Size: {len(G_agg.nodes())} ({((len(G_agg.nodes()) / len(agg_nodes)) * 100):.4f}% saturation)")

len cats: 20 -- len tracts: 1029 -- # combos: 20580


15025

### add edges and attr

In [ ]:
# create fast lookup
poi_to_bucket = {}
for bucket, poi_list in mapping.items():
    for poi in poi_list:
        poi_to_bucket[poi] = bucket

# add edges from original G based on lookup (incl attrs)
for u, v, attrs in G.edges(data=True):
    bucket_u = poi_to_bucket.get(u)
    bucket_v = poi_to_bucket.get(v)

    if bucket_u and bucket_v and (bucket_u != bucket_v):
        cur_weight = 0
        cur_cov = 0
        new_cov = attrs.get('N_COVISITS', 0)
        # check if already exists
        if G_agg.has_edge(bucket_u, bucket_v):
            cur_weight = G_agg[bucket_u][bucket_v].get('weight', 0)
            cur_cov = G_agg[bucket_u][bucket_v].get('N_COVISITS', 0)

        # upsert (create or override)
        G_agg.add_edge(bucket_u, bucket_v, weight=cur_weight+1, N_COVISITS=cur_cov+new_cov)

# --- recompute dep ---
for i, j, attrs in G_agg.edges(data=True):
    cv = attrs.get('N_COVISITS')
    visits_i = G_agg.nodes[i]['total_visits']
    visits_j = G_agg.nodes[j]['total_visits']
    dep = cv / (visits_i * visits_j)
    # round small dep to 0
    if dep >= 10**-3:
        attrs['DEP'] = 0
    else:
        attrs['DEP'] = dep


# === outside attrs ===

# gather internal points of census tracts
gaz = pd.read_csv('data/geo/2020_gaz_tracts_25.txt', sep='\t', dtype={'GEOID': str})
gaz.columns = gaz.columns.str.strip()
tract_coords = gaz.set_index('GEOID')[['INTPTLAT', 'INTPTLONG']].to_dict('index')

# visit-weighted vincentization function for distribution aggregation
def weighted_vincentizer(pois, dist_key, n_bins=4):
    default = np.full(n_bins, 1.0 / n_bins)
    agg = np.zeros(n_bins)
    total_tv = 0.0
    for poi in pois:
        tv = G.nodes[poi].get('total_visits', 0)
        dist = G.nodes[poi].get(dist_key, None)
        if dist is not None and tv > 0:
            agg += np.asarray(dist) * tv # multiplies the entire array at once
            total_tv += tv
    return agg / total_tv if total_tv > 0 else default


# nodes (poi_type + lat/lon + time + income)
for node, attrs in G_agg.nodes(data=True):
    c, t = node.split('||')
    attrs['poi_type'] = c

    if t in tract_coords:
        attrs['latitude'] = float(tract_coords[t]['INTPTLAT'])
        attrs['longitude'] = float(tract_coords[t]['INTPTLONG'])
    else:
        attrs['latitude'] = None
        attrs['longitude'] = None

    # --- time/income ---

    pois = mapping[node]
    
    # time
    t_quantiles = ['0', '6', '12', '18']
    attrs['time_dist'] = weighted_vincentizer(pois, 'time_dist')

    # income
    inc_quantiles = ['1', '2', '3', '4']
    attrs['inc_dist'] = weighted_vincentizer(pois, 'inc_dist')

# edges (distance)
for i, j, attrs in G_agg.edges(data=True):
    lat1, lon1 = G_agg.nodes[i].get('latitude'), G_agg.nodes[i].get('longitude')
    lat2, lon2 = G_agg.nodes[j].get('latitude'), G_agg.nodes[j].get('longitude')

    if None not in (lat1, lon1, lat2, lon2):
        attrs['DIST_KM'] = distance((lat1, lon1), (lat2, lon2)).km
    else:
        attrs['DIST_KM'] = None

# save
with open('data/agg/agg_network_24h_dir.pkl', 'wb') as f:
    pickle.dump(G_agg, f)
print(f'Saved with {len(G_agg.nodes())} nodes and {len(G_agg.edges())} edges')

Saved with 15025 nodes and 10479712 edges
